# Triagem de resíduos recicláveis

Este notebook contém o código de preparação, treino e avaliação em células Python legíveis. Execute as células em ordem. Para usar GPU, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.


In [ ]:
%pip install -q "tensorflow>=2.21,<2.22" "Pillow>=10,<12" "matplotlib>=3.8,<4"


## Preparação dos dados

As classes públicas de papel e papelão são reunidas em `papel`; `trash` é excluída. Arquivos idênticos com rótulos conflitantes são removidos e imagens semelhantes permanecem no mesmo conjunto.


In [ ]:
"""Preparação determinística dos dados públicos e locais."""

from __future__ import annotations

import csv
import hashlib
import json
import random
import urllib.request
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image, ImageOps

CLASSES = ("papel", "plastico", "vidro", "metal")
TRASHNET_LABELS = {
    "paper": "papel",
    "cardboard": "papel",
    "plastic": "plastico",
    "glass": "vidro",
    "metal": "metal",
}
TRASHNET_URL = "https://github.com/garythung/trashnet/raw/refs/heads/master/data/dataset-resized.zip"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}
MANIFEST_FIELDS = ("path", "label", "source", "split", "sha256", "group")


def download_trashnet(destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        urllib.request.urlretrieve(TRASHNET_URL, destination)
    if not zipfile.is_zipfile(destination):
        raise ValueError(f"Arquivo não é ZIP válido: {destination}")
    return destination


def extract_trashnet(archive: Path, destination: Path) -> Path:
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zipped:
        for entry in zipped.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError(f"Caminho inseguro no ZIP: {entry.filename}")
        zipped.extractall(destination)
    roots = [
        path for path in destination.rglob("dataset-resized")
        if path.is_dir() and all((path / original).is_dir() for original in TRASHNET_LABELS)
    ]
    if len(roots) != 1:
        raise ValueError("ZIP precisa conter exatamente uma pasta dataset-resized")
    return roots[0]


def image_fingerprint(path: Path) -> tuple[str, int]:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    with Image.open(path) as opened:
        image = ImageOps.exif_transpose(opened).convert("L").resize((9, 8))
        pixels = list(image.getdata())
    value = 0
    for row in range(8):
        for column in range(8):
            value = (value << 1) | (pixels[row * 9 + column] > pixels[row * 9 + column + 1])
    return digest, value


def scan_public(root: Path) -> list[dict[str, str | int]]:
    records: list[dict[str, str | int]] = []
    for original, label in TRASHNET_LABELS.items():
        folder = root / original
        if not folder.is_dir():
            raise ValueError(f"Classe TrashNet ausente: {folder}")
        for path in sorted(folder.iterdir()):
            if path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            sha, dhash = image_fingerprint(path)
            records.append({"path": str(path.resolve()), "label": label, "source": "trashnet", "sha256": sha, "dhash": dhash})
    if not records:
        raise ValueError("Nenhuma imagem pública encontrada")
    return records


def deduplicate_public(records: list[dict[str, str | int]]) -> tuple[list[dict[str, str | int]], list[dict[str, str]]]:
    by_hash: dict[str, list[dict[str, str | int]]] = defaultdict(list)
    for record in records:
        by_hash[str(record["sha256"])].append(record)
    kept: list[dict[str, str | int]] = []
    excluded: list[dict[str, str]] = []
    for group in by_hash.values():
        if len({record["label"] for record in group}) > 1:
            for record in group:
                excluded.append({"path": str(record["path"]), "label": str(record["label"]), "sha256": str(record["sha256"]), "reason": "conflicting_labels"})
        else:
            kept.append(group[0])
            for record in group[1:]:
                excluded.append({"path": str(record["path"]), "label": str(record["label"]), "sha256": str(record["sha256"]), "reason": "exact_duplicate"})
    return kept, excluded


def scan_local(root: Path) -> list[dict[str, str | int]]:
    labels_file = root / "labels.csv"
    if not labels_file.exists():
        raise ValueError(f"Anotações locais ausentes: {labels_file}")
    records: list[dict[str, str | int]] = []
    seen: set[Path] = set()
    seen_hashes: set[str] = set()
    with labels_file.open(newline="", encoding="utf-8-sig") as source:
        reader = csv.DictReader(source)
        if reader.fieldnames != ["path", "label"]:
            raise ValueError("labels.csv precisa ter cabeçalho: path,label")
        for line, row in enumerate(reader, start=2):
            relative = Path(row["path"])
            path = (root / relative).resolve()
            if relative.is_absolute() or not path.is_relative_to(root.resolve()):
                raise ValueError(f"Linha {line}: caminho fora da pasta local")
            if path in seen or not path.is_file() or path.suffix.lower() not in IMAGE_SUFFIXES:
                raise ValueError(f"Linha {line}: imagem ausente, repetida ou formato inválido: {relative}")
            if row["label"] not in CLASSES:
                raise ValueError(f"Linha {line}: rótulo inválido: {row['label']}")
            seen.add(path)
            sha, dhash = image_fingerprint(path)
            if sha in seen_hashes:
                raise ValueError(f"Linha {line}: foto local duplicada: {relative}")
            seen_hashes.add(sha)
            records.append({"path": str(path), "label": row["label"], "source": "local", "split": "local_test", "sha256": sha, "dhash": dhash, "group": f"local-{line}"})
    if not records:
        raise ValueError("labels.csv não contém imagens")
    if set(record["label"] for record in records) != set(CLASSES):
        raise ValueError("O teste local precisa conter as quatro classes")
    return records


def group_public(records: list[dict[str, str | int]], max_distance: int = 4) -> None:
    """Agrupa fotos visualmente próximas da mesma classe antes da divisão."""
    parent = list(range(len(records)))

    def find(index: int) -> int:
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    def union(a: int, b: int) -> None:
        parent[find(a)] = find(b)

    by_label: dict[str, list[int]] = defaultdict(list)
    sha_labels: dict[str, str | int] = {}
    for index, record in enumerate(records):
        sha = str(record["sha256"])
        if sha in sha_labels and sha_labels[sha] != record["label"]:
            raise ValueError("Imagem idêntica em classes públicas diferentes")
        sha_labels[sha] = record["label"]
        by_label[str(record["label"])].append(index)

    for indices in by_label.values():
        for position, left in enumerate(indices):
            for right in indices[position + 1 :]:
                if records[left]["sha256"] == records[right]["sha256"] or (int(records[left]["dhash"]) ^ int(records[right]["dhash"])).bit_count() <= max_distance:
                    union(left, right)
    for index, record in enumerate(records):
        record["group"] = f"{record['label']}-{find(index)}"


def split_public(records: list[dict[str, str | int]], seed: int = 42) -> None:
    randomizer = random.Random(seed)
    by_label: dict[str, dict[str, list[dict[str, str | int]]]] = defaultdict(lambda: defaultdict(list))
    for record in records:
        by_label[str(record["label"])][str(record["group"])].append(record)

    for label in CLASSES:
        groups = list(by_label[label].values())
        if len(groups) < 3:
            raise ValueError(f"Poucos grupos independentes na classe {label}")
        randomizer.shuffle(groups)
        groups.sort(key=len, reverse=True)
        total = sum(map(len, groups))
        targets = {"train": 0.70 * total, "val": 0.15 * total, "public_test": 0.15 * total}
        counts = dict.fromkeys(targets, 0)
        for group in groups:
            split = max(targets, key=lambda name: (targets[name] - counts[name]) / targets[name])
            for record in group:
                record["split"] = split
            counts[split] += len(group)
        if any(counts[name] == 0 for name in targets):
            raise ValueError(f"Divisão sem exemplos de {label} em algum conjunto")


def write_manifest(records: list[dict[str, str | int]], destination: Path, seed: int, excluded: list[dict[str, str]]) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    with (destination / "manifest.csv").open("w", newline="", encoding="utf-8") as target:
        writer = csv.DictWriter(target, fieldnames=MANIFEST_FIELDS)
        writer.writeheader()
        writer.writerows({field: record[field] for field in MANIFEST_FIELDS} for record in records)
    with (destination / "excluded_public.csv").open("w", newline="", encoding="utf-8") as target:
        writer = csv.DictWriter(target, fieldnames=("path", "label", "sha256", "reason"))
        writer.writeheader()
        writer.writerows(excluded)
    counts: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))
    for record in records:
        counts[str(record["split"])][str(record["label"])] += 1
    (destination / "data_summary.json").write_text(
        json.dumps({"seed": seed, "classes": CLASSES, "counts": counts, "excluded_public": dict(Counter(row["reason"] for row in excluded)), "trashnet_url": TRASHNET_URL}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def read_manifest(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as source:
        rows = list(csv.DictReader(source))
    if not rows or set(rows[0]) != set(MANIFEST_FIELDS):
        raise ValueError("Manifesto ausente ou inválido")
    for row in rows:
        if row["label"] not in CLASSES or row["split"] not in {"train", "val", "public_test", "local_test"}:
            raise ValueError("Rótulo ou divisão inválida no manifesto")
        if not Path(row["path"]).is_file():
            raise FileNotFoundError(row["path"])
    return rows


def prepare(archive: Path, local_dir: Path | None, output: Path, seed: int = 42) -> None:
    if not archive.exists():
        download_trashnet(archive)
    public_root = extract_trashnet(archive, output / "trashnet")
    public, excluded = deduplicate_public(scan_public(public_root))
    local = scan_local(local_dir) if local_dir is not None else []
    public_hashes = {str(record["sha256"]) for record in public}
    if any(record["sha256"] in public_hashes for record in local):
        raise ValueError("Uma imagem local é idêntica a uma imagem pública")
    group_public(public)
    split_public(public, seed)
    write_manifest(public + local, output, seed, excluded)


## Modelo e métricas

A MobileNetV2 é treinada em duas etapas. A seleção usa apenas a validação pública. As fotos próprias entram somente no teste local.


In [ ]:
"""Treino e avaliação da MobileNetV2 no Colab."""

from __future__ import annotations

import csv
import json
import random
from collections import Counter
from pathlib import Path


IMAGE_SIZE = (224, 224)


def _dataset(rows: list[dict[str, str]], batch_size: int, training: bool = False, seed: int = 42):
    import numpy as np
    import tensorflow as tf
    from PIL import Image, ImageOps

    paths = [row["path"] for row in rows]
    labels = [CLASSES.index(row["label"]) for row in rows]
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(rows), seed=seed, reshuffle_each_iteration=True)

    def decode(path):
        with Image.open(path.numpy().decode("utf-8")) as opened:
            image = ImageOps.exif_transpose(opened).convert("RGB")
            image = image.resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
            return np.asarray(image, dtype=np.float32)

    def load(path, label):
        image = tf.py_function(decode, [path], Tout=tf.float32)
        image.set_shape((*IMAGE_SIZE, 3))
        return image, label

    return dataset.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)


def _build_model():
    import tensorflow as tf

    augmentation = tf.keras.Sequential(
        [
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.08),
            tf.keras.layers.RandomZoom(0.10),
        ],
        name="train_augmentation",
    )
    base = tf.keras.applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3), include_top=False, weights="imagenet"
    )
    base.trainable = False
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = augmentation(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax")(x)
    return tf.keras.Model(inputs, outputs), base


def train(manifest: Path, output: Path, seed: int = 42, batch_size: int = 32, head_epochs: int = 12, fine_epochs: int = 12) -> None:
    import numpy as np
    import tensorflow as tf

    if head_epochs < 1 or fine_epochs < 1 or batch_size < 1:
        raise ValueError("Épocas e tamanho do lote devem ser positivos")
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except RuntimeError:
        pass

    rows = read_manifest(manifest)
    train_rows = [row for row in rows if row["split"] == "train"]
    val_rows = [row for row in rows if row["split"] == "val"]
    if not train_rows or not val_rows:
        raise ValueError("Manifesto sem treino ou validação")
    output.mkdir(parents=True, exist_ok=True)
    train_data = _dataset(train_rows, batch_size, training=True, seed=seed)
    val_data = _dataset(val_rows, batch_size)
    counts = Counter(row["label"] for row in train_rows)
    class_weight = {index: len(train_rows) / (len(CLASSES) * counts[label]) for index, label in enumerate(CLASSES)}

    model, base = _build_model()
    loss = tf.keras.losses.SparseCategoricalCrossentropy()
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
    head_path = output / "head.weights.h5"
    fine_path = output / "fine.weights.h5"

    def callbacks(path: Path):
        return [
            tf.keras.callbacks.ModelCheckpoint(str(path), monitor="val_loss", save_best_only=True, save_weights_only=True),
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        ]

    head_history = model.fit(train_data, validation_data=val_data, epochs=head_epochs, class_weight=class_weight, callbacks=callbacks(head_path))
    model.load_weights(head_path)

    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers[-30:]:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=loss, metrics=["accuracy"])
    fine_history = model.fit(train_data, validation_data=val_data, epochs=fine_epochs, class_weight=class_weight, callbacks=callbacks(fine_path))

    head_best = min(head_history.history["val_loss"])
    fine_best = min(fine_history.history["val_loss"])
    selected = "fine" if fine_best < head_best else "head"
    model.load_weights(fine_path if selected == "fine" else head_path)
    model.save(output / "model.keras")
    (output / "training.json").write_text(
        json.dumps(
            {
                "seed": seed,
                "classes": CLASSES,
                "image_size": IMAGE_SIZE,
                "batch_size": batch_size,
                "class_weight": class_weight,
                "tensorflow_version": tf.__version__,
                "selected_stage": selected,
                "head": head_history.history,
                "fine": fine_history.history,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


def _metrics(actual: list[int], predicted: list[int]) -> dict:
    matrix = [[0 for _ in CLASSES] for _ in CLASSES]
    for truth, guess in zip(actual, predicted, strict=True):
        matrix[truth][guess] += 1
    per_class = {}
    for index, label in enumerate(CLASSES):
        tp = matrix[index][index]
        support = sum(matrix[index])
        predicted_count = sum(row[index] for row in matrix)
        precision = tp / predicted_count if predicted_count else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[label] = {"support": support, "correct": tp, "precision": precision, "recall": recall, "f1": f1}
    return {
        "n": len(actual),
        "accuracy": sum(matrix[i][i] for i in range(len(CLASSES))) / len(actual),
        "macro_f1": sum(item["f1"] for item in per_class.values()) / len(CLASSES),
        "per_class": per_class,
        "confusion_matrix": matrix,
    }


def _save_figure(rows: list[dict[str, str]], predictions: list[int], split: str, output: Path, matrix: list[list[int]]) -> None:
    import matplotlib.pyplot as plt
    from PIL import Image, ImageOps

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(len(CLASSES)), CLASSES)
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("Predição")
    ax.set_ylabel("Rótulo real")
    ax.set_title(split)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, str(matrix[i][j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(output / f"confusion_{split}.png", dpi=150)
    plt.close(fig)

    pairs = list(zip(rows, predictions, strict=True))
    errors = [(row, guess) for row, guess in pairs if CLASSES[guess] != row["label"]]
    correct = [(row, guess) for row, guess in pairs if CLASSES[guess] == row["label"]]
    examples = errors[:6] + correct[:6]
    if not examples:
        return
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for ax in axes.flat:
        ax.axis("off")
    for ax, (row, guess) in zip(axes.flat, examples, strict=False):
        with Image.open(row["path"]) as image:
            ax.imshow(ImageOps.exif_transpose(image).convert("RGB"))
        ax.set_title(f"Real: {row['label']}\nPred.: {CLASSES[guess]}", fontsize=9)
    fig.tight_layout()
    fig.savefig(output / f"examples_{split}.png", dpi=150)
    plt.close(fig)


def evaluate(manifest: Path, model_path: Path, output: Path, batch_size: int = 32) -> None:
    import tensorflow as tf

    rows = read_manifest(manifest)
    model = tf.keras.models.load_model(model_path)
    output.mkdir(parents=True, exist_ok=True)
    summary = {}
    for split in ("public_test", "local_test"):
        subset = [row for row in rows if row["split"] == split]
        if not subset:
            if split == "local_test":
                continue
            raise ValueError(f"Conjunto de teste vazio: {split}")
        probabilities = model.predict(_dataset(subset, batch_size), verbose=0)
        predictions = probabilities.argmax(axis=1).tolist()
        actual = [CLASSES.index(row["label"]) for row in subset]
        result = _metrics(actual, predictions)
        summary[split] = result
        with (output / f"predictions_{split}.csv").open("w", newline="", encoding="utf-8") as target:
            writer = csv.writer(target)
            writer.writerow(("path", "true", "predicted", "confidence", "correct"))
            for row, guess, probs in zip(subset, predictions, probabilities, strict=True):
                writer.writerow((row["path"], row["label"], CLASSES[guess], float(probs[guess]), CLASSES[guess] == row["label"]))
        _save_figure(subset, predictions, split, output, result["confusion_matrix"])
    (output / "metrics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")


## Fotos próprias e divisão

Se ainda não tiver as fotos, mantenha `USAR_FOTOS_LOCAIS = False`. Quando tiver um ZIP com `labels.csv` e as fotos referenciadas nele, mude para `True` e execute esta célula.


In [ ]:
from google.colab import files

USAR_FOTOS_LOCAIS = False
local_dir = None
if USAR_FOTOS_LOCAIS:
    print('Selecione o ZIP com labels.csv e as fotos próprias:')
    uploaded = files.upload()
    assert len(uploaded) == 1 and next(iter(uploaded)).lower().endswith('.zip')
    with zipfile.ZipFile(next(iter(uploaded))) as archive:
        archive.extractall('/content/runs/local')
    labels_files = list(Path('/content/runs/local').rglob('labels.csv'))
    assert len(labels_files) == 1, 'O ZIP precisa conter exatamente um labels.csv'
    local_dir = labels_files[0].parent

prepare(Path('/content/data/dataset-resized.zip'), local_dir, Path('/content/runs/data'))
print(Path('/content/runs/data/data_summary.json').read_text())


## Treino

O treino salva os pesos de cada etapa e o modelo escolhido pela validação.


In [ ]:
train(Path('/content/runs/data/manifest.csv'), Path('/content/runs/model'))


## Avaliação

São gerados métricas, previsões, matrizes de confusão e exemplos visuais. Sem fotos próprias, apenas o teste público é avaliado.


In [ ]:
evaluate(Path('/content/runs/data/manifest.csv'), Path('/content/runs/model/model.keras'), Path('/content/runs/evaluation'))
print(Path('/content/runs/evaluation/metrics.json').read_text())


## Baixar resultados


In [ ]:
import shutil
shutil.make_archive('/content/triagem-resultados', 'zip', '/content/runs')
files.download('/content/triagem-resultados.zip')
